In [33]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import csv

# Imports centralisés — utilisez cette cellule pour gérer les dépendances


In [34]:
# Charger le second fichier avec les colonnes du CSV

df_epargne = pd.read_csv('Epargne_salariale.csv', sep=';')
df_epargne.head()

,Date,Montant (€)
0,Déjà disponible,"3 885,50"
1,01/06/2027,"1 804,21"
2,01/06/2028,"2 904,85"
3,01/06/2029,"6 108,18"
4,01/06/2030,"4 020,22"


In [35]:
# Charger aussi le tableau d'amortissement pour les cellules suivantes
df_amort = pd.read_csv('Tab_amort_pret.csv', sep=';', header=None)
df_amort.columns = ['Date', 'Capital restant dû','Capital remboursé', 'Intérêts', 'Assurance', 'Mensualité']
df_amort.head()

,Date,Capital restant dû,Capital remboursé,Intérêts,Assurance,Mensualité
0,05/12/2025,132 303,821.04,190.74,61.64,1073.42
1,05/01/2026,131 482,822.23,189.55,61.64,1073.42
2,05/02/2026,130 660,823.41,188.37,61.64,1073.42
3,05/03/2026,129 836,824.60,187.18,61.64,1073.42
4,05/04/2026,129 012,825.79,185.99,61.64,1073.42


In [36]:
# Préparer les données ETF

# valeurs initiales
initial_amount = 30000
start_date = pd.Timestamp('2024-02-05')
end_date = pd.Timestamp('2037-12-05')

# Générer une série mensuelle sur le jour 5 de chaque mois
monthly_starts = pd.date_range(start=start_date, end=end_date, freq='MS')
monthly_dates = monthly_starts + pd.DateOffset(days=start_date.day - 1)
# s'assurer que la date de début et de fin sont incluses
if monthly_dates[0] != start_date:
    monthly_dates = monthly_dates.insert(0, start_date)
if monthly_dates[-1] != end_date:
    monthly_dates = monthly_dates.append(pd.DatetimeIndex([end_date]))
monthly_dates = pd.DatetimeIndex(sorted(set(monthly_dates)))

rates = [0.06, 0.08, 0.10, 0.12, 0.14, 0.16]

rows = []
for rate in rates:
    values = [initial_amount]
    for i in range(1, len(monthly_dates)):
        prev_date = monthly_dates[i - 1]
        curr_date = monthly_dates[i]
        months = (curr_date.year - prev_date.year) * 12 + (curr_date.month - prev_date.month)
        # avec nos dates sur le même jour du mois, months vaudra 1 pour pas mal d'étapes
        values.append(values[-1] * (1 + rate / 12) ** months)
    rows.append(pd.DataFrame({
        'Date': monthly_dates,
        'Montant': values,
        'Taux': [f'{rate * 100:.0f}%'] * len(monthly_dates)
    }))

etf_growth = pd.concat(rows, ignore_index=True)

months_total = (end_date.year - start_date.year) * 12 + (end_date.month - start_date.month)
summary = pd.DataFrame({
    'Taux': [f'{rate * 100:.0f}%' for rate in rates],
    'Montant final (€)': [round(initial_amount * (1 + rate / 12) ** months_total, 2) for rate in rates],
    'Gain (€)': [round(initial_amount * (1 + rate / 12) ** months_total - initial_amount, 2) for rate in rates]
})

summary

,Taux,Montant final (€),Gain (€)
0,6%,68657.42,38657.42
1,8%,90395.23,60395.23
2,10%,118961.35,88961.35
3,12%,156483.77,126483.77
4,14%,205748.37,175748.37
5,16%,270400.72,240400.72


In [39]:
# Préparer le capital restant dû
amort_curve = df_amort[['Date', 'Capital restant dû']].copy()
amort_curve['Date'] = pd.to_datetime(amort_curve['Date'], dayfirst=True)
amort_curve = amort_curve.sort_values('Date')
amort_curve['capital restant dû'] = pd.to_numeric(
    amort_curve['Capital restant dû'].astype(str).str.replace(' ', '').str.replace(',', '.'),
    errors='coerce'
)

# Préparer etf_growth
etf_growth['Date'] = pd.to_datetime(etf_growth['Date'])

# Graphique comparatif
fig = px.line(
    etf_growth,
    x='Date',
    y='Montant',
    color='Taux',
    labels={'Date': 'Date', 'Montant': 'Montant (€)', 'Taux': 'Taux de rendement'},
    title='Évolution mensuelle des ETF et du capital restant dû',
    height=700
)

# Fond noir et texte en blanc pour lisibilité
fig.update_layout(
    paper_bgcolor='black',
    plot_bgcolor='black',
    font_color='white',
)

# Axes et grilles lisibles sur fond sombre
fig.update_xaxes(showgrid=True, gridcolor='gray', tickfont_color='white', title_font_color='white')
fig.update_yaxes(showgrid=True, gridcolor='gray', tickfont_color='white', title_font_color='white')

# Ajouter la courbe du capital restant dû
fig.add_scatter(
    x=amort_curve['Date'],
    y=amort_curve['capital restant dû'],
    mode='lines',
    name='Capital restant dû',
    line=dict(color='white', width=3)
)

# Construire un index mensuel correspondant aux dates d'etf_growth
monthly_index = pd.DatetimeIndex(sorted(etf_growth['Date'].unique()))
# Interpoler le capital restant dû sur ces dates
amort_series = amort_curve.set_index('Date')['capital restant dû']
amort_on_months = amort_series.reindex(monthly_index).interpolate(method='time')

# Récupérer les couleurs assignées par plotly pour chaque trace ETF
color_map = {}
for trace in fig.data:
    name = trace.name
    # seules les traces ETF ont des noms correspondant aux taux (ex: '6%')
    if isinstance(name, str) and name.endswith('%'):
        c = None
        if hasattr(trace, 'line') and getattr(trace.line, 'color', None) is not None:
            c = trace.line.color
        color_map[name] = c

# Calculer intersections (première date où ETF >= capital restant dû) et annoter
annotations = []
for taux, grp in etf_growth.groupby('Taux'):
    grp_sorted = grp.sort_values('Date').set_index('Date')
    etf_vals = grp_sorted['Montant']
    amort_vals = amort_on_months.reindex(grp_sorted.index).interpolate(method='time')
    mask = (etf_vals >= amort_vals).fillna(False)
    if mask.any():
        inter_date = mask.idxmax()
        inter_val = float(etf_vals.loc[inter_date])
        color = color_map.get(taux, 'yellow')
        # ajouter un marqueur avec la même couleur que la courbe si disponible
        fig.add_scatter(x=[inter_date], y=[inter_val], mode='markers', marker=dict(size=10, color=color), name=f'Intersection {taux}')
        # annotation texte avec date
        annotations.append(dict(
            x=inter_date,
            y=inter_val,
            xanchor='left',
            yanchor='bottom',
            text=f"{taux} — {inter_date.strftime('%d/%m/%Y')}",
            font=dict(color='white'),
            showarrow=True,
            arrowcolor=color
        ))

# Ajouter une ligne verticale pointillée rouge au 05/09/2031
vline_date = pd.Timestamp('2031-09-05')
fig.add_vline(x=vline_date, line=dict(color='red', width=2, dash='dot'))
# Annotation pour la ligne verticale
annotations.append(dict(
    x=vline_date,
    y=1.02 * etf_growth['Montant'].max(),
    xanchor='left',
    yanchor='bottom',
    text='Entrée Rose et Célestin supérieur — 05/09/2031',
    font=dict(color='red'),
    showarrow=False
))

fig.update_layout(annotations=annotations)

fig.show()

In [46]:
# Ajouter les montants d’épargne salariale aux courbes ETF aux dates appropriées

# Préparer les versements d’épargne salariale
if 'df_epargne' not in globals():
    df_epargne = pd.read_csv('Epargne_salariale.csv', sep=';')

savings = df_epargne.copy()
savings['Date'] = pd.to_datetime(savings['Date'], dayfirst=True, errors='coerce')
savings['Montant (€)'] = savings['Montant (€)'].astype(str).str.replace(' ', '').str.replace(',', '.')
savings['Montant (€)'] = pd.to_numeric(savings['Montant (€)'], errors='coerce')

# Ajouter explicitement le « déjà disponible » à la date demandée
savings.loc[savings['Date'].isna(), 'Date'] = pd.Timestamp('2026-08-05')
savings = savings.dropna(subset=['Date', 'Montant (€)']).sort_values('Date').reset_index(drop=True)

# Construire un dataframe combiné ETF + épargne salariale
rows = []
for rate in rates:
    taux_label = f'{rate * 100:.0f}%'
    grp = etf_growth[etf_growth['Taux'] == taux_label].sort_values('Date').copy()
    values = grp['Montant'].astype(float).to_numpy(copy=True)

    for _, dep in savings[['Date', 'Montant (€)']].dropna().iterrows():
        dep_date = dep['Date']
        dep_amount = float(dep['Montant (€)'])
        if dep_date > end_date:
            continue
        idx = pd.Index(grp['Date']).get_indexer([dep_date], method='nearest')
        if len(idx) and idx[0] != -1:
            values[idx[0]:] = values[idx[0]:] + dep_amount

    rows.append(pd.DataFrame({
        'Date': grp['Date'],
        'Montant total disponible': values,
        'Taux': [taux_label] * len(grp)
    }))

combined_df = pd.concat(rows, ignore_index=True)
combined_df['Date'] = pd.to_datetime(combined_df['Date'])
combined_df = combined_df.sort_values(['Taux', 'Date']).reset_index(drop=True)

# Tracer le graphe combiné
fig_combined = px.line(
    combined_df,
    x='Date',
    y='Montant total disponible',
    color='Taux',
    labels={'Date': 'Date', 'Montant total disponible': 'Capital total (€)', 'Taux': 'Taux de rendement'},
    title='ETF + Épargne salariale disponible selon le taux de rendement',
    height=700
)

fig_combined.update_layout(
    paper_bgcolor='black',
    plot_bgcolor='black',
    font_color='white',
)
fig_combined.update_xaxes(showgrid=True, gridcolor='gray', tickfont_color='white', title_font_color='white')
fig_combined.update_yaxes(showgrid=True, gridcolor='gray', tickfont_color='white', title_font_color='white')

# Ajouter la courbe du capital restant dû
amort_curve = df_amort[['Date', 'Capital restant dû']].copy()
amort_curve['Date'] = pd.to_datetime(amort_curve['Date'], dayfirst=True)
amort_curve = amort_curve.sort_values('Date')
amort_curve['capital restant dû'] = pd.to_numeric(
    amort_curve['Capital restant dû'].astype(str).str.replace(' ', '').str.replace(',', '.'),
    errors='coerce'
)
fig_combined.add_scatter(
    x=amort_curve['Date'],
    y=amort_curve['capital restant dû'],
    mode='lines',
    name='Capital restant dû',
    line=dict(color='white', width=3)
)

# Calculer les intersections et conserver la ligne verticale
combined_dates = pd.DatetimeIndex(sorted(combined_df['Date'].unique()))
amort_series = amort_curve.set_index('Date')['capital restant dû']
amort_on_dates = amort_series.reindex(combined_dates).interpolate(method='time')

annotations = []
intersections = []
color_map = {}
for trace in fig_combined.data:
    name = trace.name
    if isinstance(name, str) and name.endswith('%'):
        color = getattr(getattr(trace, 'line', None), 'color', None)
        color_map[name] = color or 'yellow'

for taux, grp in combined_df.groupby('Taux'):
    grp_sorted = grp.sort_values('Date').set_index('Date')
    combined_vals = grp_sorted['Montant total disponible']
    amort_vals = amort_on_dates.reindex(grp_sorted.index).interpolate(method='time')
    mask = (combined_vals >= amort_vals).fillna(False)
    if mask.any():
        inter_date = mask[mask].index[0]
        inter_val = float(combined_vals.loc[inter_date])
        color = color_map.get(taux, 'yellow')
        fig_combined.add_scatter(
            x=[inter_date],
            y=[inter_val],
            mode='markers',
            marker=dict(size=10, color=color),
            name=f'Intersection {taux}'
        )
        annotations.append(dict(
            x=inter_date,
            y=inter_val,
            xanchor='left',
            yanchor='bottom',
            text=f"{taux} — {inter_date.strftime('%d/%m/%Y')}",
            font=dict(color='white'),
            showarrow=True,
            arrowcolor=color
        ))
        intersections.append({
            'Taux': taux,
            "Date d'intersection": inter_date,
            'Montant total disponible': inter_val,
            'Capital restant dû': float(amort_vals.loc[inter_date])
        })

vline_date = pd.Timestamp('2031-09-05')
fig_combined.add_vline(x=vline_date, line=dict(color='red', width=2, dash='dot'))
annotations.append(dict(
    x=vline_date,
    y=1.02 * combined_df['Montant total disponible'].max(),
    xanchor='left',
    yanchor='bottom',
    text='Entrée Rose et Célestin supérieur — 05/09/2031',
    font=dict(color='red'),
    showarrow=False
))

fig_combined.update_layout(annotations=annotations)

intersections_df = pd.DataFrame(intersections).sort_values("Date d'intersection")
fig_combined.show()
intersections_df

/tmp/ipykernel_63996/881592249.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  savings['Date'] = pd.to_datetime(savings['Date'], dayfirst=True, errors='coerce')


,Taux,Date d'intersection,Montant total disponible,Capital restant dû
3,16%,2030-01-05,91533.761234,90648.0
2,14%,2030-06-05,91161.009416,86230.0
1,12%,2030-09-05,84565.986253,83563.0
0,10%,2031-03-05,79462.544618,78196.0
5,8%,2031-06-05,75963.030795,75495.0
4,6%,2031-12-05,70072.036201,70058.0


In [47]:
# Ajouter les versements d’épargne salariale avec rendement ETF selon le taux associé

# Préparer les versements d’épargne salariale
if 'df_epargne' not in globals():
    df_epargne = pd.read_csv('Epargne_salariale.csv', sep=';')

savings = df_epargne.copy()
savings['Date'] = pd.to_datetime(savings['Date'], dayfirst=True, errors='coerce')
savings['Montant (€)'] = savings['Montant (€)'].astype(str).str.replace(' ', '').str.replace(',', '.')
savings['Montant (€)'] = pd.to_numeric(savings['Montant (€)'], errors='coerce')
savings.loc[savings['Date'].isna(), 'Date'] = pd.Timestamp('2026-08-05')
savings = savings.dropna(subset=['Date', 'Montant (€)']).sort_values('Date').reset_index(drop=True)

# Construire les séries ETF + épargne salariale avec rendement propre à chaque taux
combined_rows = []
for rate in rates:
    taux_label = f'{rate * 100:.0f}%'
    grp = etf_growth[etf_growth['Taux'] == taux_label].sort_values('Date').copy()
    grp = grp.reset_index(drop=True)

    values = grp['Montant'].astype(float).to_numpy(copy=True)

    for _, dep in savings.iterrows():
        dep_date = dep['Date']
        dep_amount = float(dep['Montant (€)'])
        if dep_date > end_date:
            continue

        pos = None
        for j, d in enumerate(grp['Date']):
            if d >= dep_date:
                pos = j
                break
        if pos is None:
            continue

        for j in range(pos, len(values)):
            months = (grp['Date'].iloc[j].year - dep_date.year) * 12 + (grp['Date'].iloc[j].month - dep_date.month)
            if months < 0:
                months = 0
            values[j] += dep_amount * (1 + rate / 12) ** months

    combined_rows.append(pd.DataFrame({
        'Date': grp['Date'],
        'Montant total disponible': values,
        'Taux': [taux_label] * len(grp)
    }))

combined_df = pd.concat(combined_rows, ignore_index=True)
combined_df['Date'] = pd.to_datetime(combined_df['Date'])
combined_df = combined_df.sort_values(['Taux', 'Date']).reset_index(drop=True)

# Tracer le graphe final
fig_combined = px.line(
    combined_df,
    x='Date',
    y='Montant total disponible',
    color='Taux',
    labels={'Date': 'Date', 'Montant total disponible': 'Capital total (€)', 'Taux': 'Taux de rendement'},
    title='ETF + Épargne salariale disponible selon le taux de rendement',
    height=700
)

fig_combined.update_layout(
    paper_bgcolor='black',
    plot_bgcolor='black',
    font_color='white',
)
fig_combined.update_xaxes(showgrid=True, gridcolor='gray', tickfont_color='white', title_font_color='white')
fig_combined.update_yaxes(showgrid=True, gridcolor='gray', tickfont_color='white', title_font_color='white')

# Ajouter la courbe du capital restant dû
amort_curve = df_amort[['Date', 'Capital restant dû']].copy()
amort_curve['Date'] = pd.to_datetime(amort_curve['Date'], dayfirst=True)
amort_curve = amort_curve.sort_values('Date')
amort_curve['capital restant dû'] = pd.to_numeric(
    amort_curve['Capital restant dû'].astype(str).str.replace(' ', '').str.replace(',', '.'),
    errors='coerce'
)
fig_combined.add_scatter(
    x=amort_curve['Date'],
    y=amort_curve['capital restant dû'],
    mode='lines',
    name='Capital restant dû',
    line=dict(color='white', width=3)
)

# Calculer intersections et conserver la ligne rouge
combined_dates = pd.DatetimeIndex(sorted(combined_df['Date'].unique()))
amort_series = amort_curve.set_index('Date')['capital restant dû']
amort_on_dates = amort_series.reindex(combined_dates).interpolate(method='time')

annotations = []
intersections = []
color_map = {}
for trace in fig_combined.data:
    name = trace.name
    if isinstance(name, str) and name.endswith('%'):
        color = getattr(getattr(trace, 'line', None), 'color', None)
        color_map[name] = color or 'yellow'

for taux, grp in combined_df.groupby('Taux'):
    grp_sorted = grp.sort_values('Date').set_index('Date')
    combined_vals = grp_sorted['Montant total disponible']
    amort_vals = amort_on_dates.reindex(grp_sorted.index).interpolate(method='time')
    mask = (combined_vals >= amort_vals).fillna(False)
    if mask.any():
        inter_date = mask[mask].index[0]
        inter_val = float(combined_vals.loc[inter_date])
        color = color_map.get(taux, 'yellow')
        fig_combined.add_scatter(
            x=[inter_date],
            y=[inter_val],
            mode='markers',
            marker=dict(size=10, color=color),
            name=f'Intersection {taux}'
        )
        annotations.append(dict(
            x=inter_date,
            y=inter_val,
            xanchor='left',
            yanchor='bottom',
            text=f"{taux} — {inter_date.strftime('%d/%m/%Y')}",
            font=dict(color='white'),
            showarrow=True,
            arrowcolor=color
        ))
        intersections.append({
            'Taux': taux,
            "Date d'intersection": inter_date,
            'Montant total disponible': inter_val,
            'Capital restant dû': float(amort_vals.loc[inter_date])
        })

vline_date = pd.Timestamp('2031-09-05')
fig_combined.add_vline(x=vline_date, line=dict(color='red', width=2, dash='dot'))
annotations.append(dict(
    x=vline_date,
    y=1.02 * combined_df['Montant total disponible'].max(),
    xanchor='left',
    yanchor='bottom',
    text='Entrée Rose et Célestin supérieur — 05/09/2031',
    font=dict(color='red'),
    showarrow=False
))

fig_combined.update_layout(annotations=annotations)

intersections_df = pd.DataFrame(intersections).sort_values("Date d'intersection")
fig_combined.show()
intersections_df

/tmp/ipykernel_63996/3020210942.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  savings['Date'] = pd.to_datetime(savings['Date'], dayfirst=True, errors='coerce')


,Taux,Date d'intersection,Montant total disponible,Capital restant dû
3,16%,2029-11-05,94149.512876,92406.0
2,14%,2030-03-05,89490.858513,88884.0
1,12%,2030-06-05,87220.331625,86230.0
0,10%,2030-11-05,82144.101414,81780.0
5,8%,2031-05-05,76725.459843,76397.0
4,6%,2031-09-05,73132.126179,72783.0
